# Uber "Find My Ride" — Primary Survey EDA

**Scope of this notebook so far: Phase 1 (Access & audit) and Phase 2 (Cleaning & data dictionary).**
Phases 3+ (quantitative EDA, qualitative coding) will be appended later.

Rules applied (see `CLAUDE.md`):
- Question wording, answer options, question types and branching are **parsed from `uber-pickup-survey/create_uber_form.gs`** (source of truth), not retyped.
- No record is deleted. Questionable records are *flagged*; every decision is written to `data/clean/Cleaning_Log.csv`.
- Multi-select answers are split against the known option list; the original text is kept, and write-in "Other" text is kept separately.
- Denominators = eligible respondents who answered that question.

Outputs: `data/clean/Cleaned_Primary_Research.csv`, `Data_Dictionary.csv`, `Cleaning_Log.csv`, `Count_Reconciliation.csv`.

In [1]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 70)

ROOT = Path.cwd() if (Path.cwd() / 'CLAUDE.md').exists() else Path.cwd().parent
RAW = ROOT / 'data' / 'raw'
CLEAN = Path(os.environ.get('UBER_CLEAN_DIR', ROOT / 'data' / 'clean'))
FORM_GS = ROOT / 'uber-pickup-survey' / 'create_uber_form.gs'
CLEAN.mkdir(parents=True, exist_ok=True)

LOG = []
def log(phase, action, column='', rows=0, detail='', resp_ids=''):
    # Every cleaning/audit decision goes here and is exported to Cleaning_Log.csv
    LOG.append(dict(phase=phase, action=action, column=column, rows_affected=int(rows),
                    detail=detail, resp_ids=resp_ids))

print('Project root:', ROOT)

Project root: D:\Crop_python\IIMA\Airtribe\Uber\TMS


## Phase 1 — Access & audit
### 1.1 Input files

In [2]:
inputs = sorted(p for p in RAW.rglob('*') if p.is_file() and p.name != '.gitkeep')
if inputs:
    display(pd.DataFrame([{'file': str(p.relative_to(ROOT)), 'size_kb': round(p.stat().st_size / 1024, 1)}
                          for p in inputs]))
else:
    print('data/raw/ is empty')

override = os.environ.get('UBER_RAW_FILE')
if override:
    survey_path = Path(override)
else:
    cands = [p for p in inputs if p.parent == RAW and p.suffix.lower() in ('.csv', '.xlsx')
             and not p.name.lower().startswith('review')]
    if len(cands) > 1:
        pref = [p for p in cands if 'response' in p.name.lower()]
        cands = pref if len(pref) == 1 else cands
    if len(cands) != 1:
        raise FileNotFoundError(
            f'Missing input: expected exactly one Google Form export (CSV/XLSX) in {RAW}, found {len(cands)}: '
            f'{[p.name for p in cands]}. Export it from the responses Sheet via File -> Download -> CSV.')
    survey_path = cands[0]

if survey_path.suffix.lower() == '.xlsx':
    xl = pd.ExcelFile(survey_path)
    print('Sheets:', xl.sheet_names)
    sheet = 'Form Responses 1' if 'Form Responses 1' in xl.sheet_names else xl.sheet_names[0]
    raw = pd.read_excel(xl, sheet_name=sheet, dtype=str).fillna('')
else:
    sheet = None
    raw = pd.read_csv(survey_path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

stripped_headers = [c for c in raw.columns if c != c.strip()]
raw.columns = [c.strip() for c in raw.columns]
if stripped_headers:
    log(1, 'strip_header_whitespace', rows=0, detail=f'{len(stripped_headers)} headers had leading/trailing spaces')
log(1, 'load_input', rows=len(raw), detail=f'{survey_path.name}' + (f' [sheet {sheet}]' if sheet else ''))
print(f'Loaded {survey_path.name}: {raw.shape[0]} rows x {raw.shape[1]} columns (all read as text, blanks kept as "")')

,file,size_kb
0,data\raw\reviews\apple\apple_in_p1.json,39.6
1,data\raw\reviews\apple\apple_in_p10.json,36.7
2,data\raw\reviews\apple\apple_in_p2.json,42.0
3,data\raw\reviews\apple\apple_in_p3.json,39.1
4,data\raw\reviews\apple\apple_in_p4.json,43.1
5,data\raw\reviews\apple\apple_in_p5.json,38.9
6,data\raw\reviews\apple\apple_in_p6.json,38.6
7,data\raw\reviews\apple\apple_in_p7.json,37.8
8,data\raw\reviews\apple\apple_in_p8.json,39.7
9,data\raw\reviews\apple\apple_in_p9.json,38.7


Sheets: ['Form Responses 1', 'Summary']
Loaded Uber Ride & Pickup Experience Survey (Responses).xlsx: 27 rows x 29 columns (all read as text, blanks kept as "")


### 1.2 Shape, columns, dtypes, first 5 rows

In [3]:
print('Shape:', raw.shape)
display(pd.DataFrame({'column': raw.columns, 'dtype_as_read': raw.dtypes.astype(str).values,
                      'non_blank': [(raw[c].str.strip() != '').sum() for c in raw.columns]}))
display(raw.head())

Shape: (27, 29)


,column,dtype_as_read,non_blank
0,Timestamp,object,27
1,Consent,object,0
2,Q1. Have you used Uber for booking a ride?,object,27
3,Q2. How would you rate your overall experience with Uber?,object,26
4,Q2a. What is the main reason for your rating?,object,21
5,Q3. Have you booked an Uber from a crowded or complex location?,object,26
6,Q4. Where have you experienced difficulty finding or meeting your ...,object,26
7,"Q4a. If you remember, mention the location type or entrance where ...",object,0
8,Q5. How helpful is Uber's app in showing your pickup location and ...,object,26
9,Q5a. Is the app showing exact location of yours or near-by places?...,object,0


,Timestamp,Consent,Q1. Have you used Uber for booking a ride?,Q2. How would you rate your overall experience with Uber?,Q2a. What is the main reason for your rating?,Q3. Have you booked an Uber from a crowded or complex location?,Q4. Where have you experienced difficulty finding or meeting your Uber driver?,"Q4a. If you remember, mention the location type or entrance where the difficulty occurred.",Q5. How helpful is Uber's app in showing your pickup location and helping you find the driver?,Q5a. Is the app showing exact location of yours or near-by places? Please share your experience with app,Q6. What difficulties have you experienced when meeting your driver in crowded places?,Q6a. Please describe what happened during your most memorable difficult pickup.,Q7. How did the driver coordinate with you when finding your location was difficult?,Q7a. What did the driver say or do that helped or made the situation more difficult?,Q8. How would you describe the driver's behavior during a difficult pickup?,"Q8a. If you experienced frustration or particularly good service, what happened?",Q9. How did you feel when you had difficulty finding your driver?,Q9a. What was the biggest impact on you?,Q10. What was the outcome of your difficult pickup experience?,"Q10a. If the ride was cancelled or delayed, what was the main reason?",Q11. Has an Uber driver ever requested extra money beyond the fare shown in the app?,"Q11a. If yes, what reason did the driver give?",Q12. What has been your most positive experience with Uber?,"Q13. What improvements would you suggest to make Uber pickups easier, especially in crowded places?",Q14. Is there another ride-booking app you prefer over Uber?,Q14a. Why do you prefer that app?,Q15. Would you like to provide any additional feedback about Uber app and their services?,Q14 (non-Uber users). Which ride-booking app do you prefer?,Q14a (non-Uber users). Why do you prefer that app?
0,2026-10-14 10:04:57.567000,,"No, I have never used Uber",,,,,,,,,,,,,,,,,,,,,,,,,Ola,fghj
1,2026-10-14 10:07:29.236000,,"Yes, occasionally",4,,"Yes, once or twice",Railway station / Bus stand,,Often confusing or inaccurate,,"Pickup pin showed an incorrect or inconvenient location, Difficult...",vdsa,Asked me to move to another location,,Neutral,sav,Anxious or stressed,,Successfully met the driver without much delay,ewg,No,egwg,ewh,ewh,Namma Yatri,rhh,HW,,
2,2026-10-14 10:12:28.659000,,"Yes, frequently",2,sdd,"Yes, once or twice",Office complex / IT park,,Very helpful and accurate,,Pickup pin showed an incorrect or inconvenient location,wefg,Called and politely asked for directions,,Neutral,dcevw,Anxious or stressed,,Successfully met the driver without much delay,ew,Yes,ef,ewf,efg,Namma Yatri,ef,wef,,
3,2026-10-14 10:45:56.584000,,"Yes, frequently",4,Many of the drivers are polite and also friendly only some are ask...,"Yes, several times",My kids school area which is far from city,,Mostly helpful,,Pickup pin showed an incorrect or inconvenient location,My pick up came late during traffic time and I have to go to airpo...,Called and politely asked for directions,,Behavior varied across rides,Once the cab has arrived but we came just 5 min late then driver g...,Worried about being late,,Driver cancelled the booking,Depends on the area and time,Yes,Toll gate fees so he demanded extra,Mostly good pick up and dropping and also gentle behaviour and dro...,Google maps are not perfect so pick up location also is not accura...,No particular preference,If pick up is late or not confirmed and time taking too long then ...,I already shared about the location and the Google is not good to ...,,
4,2026-10-14 10:57:50.029000,,"Yes, frequently",3,"Arrival delays, impolite driver behaviour.","Yes, several times","Shopping mall, Office complex / IT park, Airport, Railway station ...",,Sometimes confusing or inaccurate,,"Driver could not find my exact location, Pickup pin showed an inco...",Driver cancelled the ride mid way.,"Called and politely ask

### 1.3 Questionnaire structure from `create_uber_form.gs`
Parses every `mc()` (single-select), `cb()` (multi-select), `note()` (paragraph), scale and text item, plus the Q1 branching.

In [4]:
gs = FORM_GS.read_text(encoding='utf-8')
STR = r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\""
unq = lambda s: re.sub(r'\\(.)', r'\1', s[1:-1])
strings = lambda s: [unq(m.group(0)) for m in re.finditer(STR, s)]

items = []
for m in re.finditer(rf'\b(mc|cb|note)\(\s*({STR})', gs):
    kind, title = m.group(1), unq(m.group(2))
    opts, req, other = [], False, False
    tail = re.match(r'\s*,\s*\[(.*?)\]\s*,\s*(true|false)(?:\s*,\s*(true|false))?', gs[m.end():], re.S)
    if kind in ('mc', 'cb') and tail:
        opts, req, other = strings(tail.group(1)), tail.group(2) == 'true', tail.group(3) == 'true'
    items.append(dict(pos=m.start(), title=title, qtype={'mc': 'single', 'cb': 'multi', 'note': 'text'}[kind],
                      options=opts, required=req, other_allowed=other))
for m in re.finditer(rf'add(Scale|Text|MultipleChoice)Item\(\)\.setTitle\(({STR})\)', gs):
    qtype = {'Scale': 'scale', 'Text': 'text', 'MultipleChoice': 'single'}[m.group(1)]
    stmt = gs[m.end():].split(';')[0]
    opts = []
    if qtype == 'scale':
        lo, hi = map(int, re.search(r'setBounds\((\d+),\s*(\d+)\)', stmt).groups())
        opts = [str(i) for i in range(lo, hi + 1)]
    items.append(dict(pos=m.start(), title=unq(m.group(2)), qtype=qtype, options=opts,
                      required='setRequired(true)' in stmt, other_allowed=False))

# Q1 choices and branching (pUber = Uber-user page, pNever = non-user page)
q1_choices = [(unq(a), b) for a, b in re.findall(rf'q1\.createChoice\(({STR}),\s*(\w+)\)', gs)]
UBER_USER_CHOICES = [c for c, page in q1_choices if page == 'pUber']
NON_USER_CHOICES = [c for c, page in q1_choices if page == 'pNever']

def qcode(title):
    m = re.match(r'\s*(Q\d+[a-z]?)\b', title)
    if not m:
        return None
    return m.group(1) + ('_nonuber' if 'non-uber' in title.lower() else '')

FORM = {}
for it in sorted(items, key=lambda d: d['pos']):
    c = qcode(it['title'])
    assert c and c not in FORM, f'Unparsed or duplicate form item: {it["title"]}'
    if c == 'Q1':
        it['options'] = [ch for ch, _ in q1_choices]
    it['eligible'] = 'all' if c == 'Q1' else ('non_uber' if c.endswith('_nonuber') else 'uber_user')
    FORM[c] = it

form_tbl = pd.DataFrame([{'code': c, 'type': d['qtype'], 'required': d['required'], 'other_allowed': d['other_allowed'],
                          'eligible': d['eligible'], 'n_options': len(d['options']), 'form_wording': d['title']}
                         for c, d in FORM.items()])
print(f'{len(FORM)} questions parsed. Q1 -> Uber page: {UBER_USER_CHOICES}; -> non-user page: {NON_USER_CHOICES}')
display(form_tbl)

27 questions parsed. Q1 -> Uber page: ['Yes, frequently', 'Yes, occasionally', 'Yes, but not in the last 3 months']; -> non-user page: ['No, I have never used Uber']


,code,type,required,other_allowed,eligible,n_options,form_wording
0,Q1,single,True,False,all,4,Q1. Have you used Uber for booking a ride?
1,Q2,scale,True,False,uber_user,5,Q2. How would you rate your overall experience with Uber?
2,Q2a,text,False,False,uber_user,0,Q2a. What is the main reason for your rating?
3,Q3,single,True,False,uber_user,4,Q3. Have you booked an Uber from a crowded or complex location?
4,Q4,multi,True,True,uber_user,9,Q4. Where have you experienced difficulty finding or meeting your ...
5,Q4a,text,False,False,uber_user,0,"Q4a. If you remember, mention the location type or entrance where ..."
6,Q5,single,True,False,uber_user,5,Q5. How helpful is Uber's app in showing your pickup location and ...
7,Q5a,text,False,False,uber_user,0,"Q5a. What worked well, or what information was missing or confusing?"
8,Q6,multi,True,True,uber_user,9,Q6. What difficulties have you experienced when meeting your drive...
9,Q6a,text,False,False,uber_user,0,Q6a. Please describe what happened during your most memorable diff...


### 1.4 Map export columns to form questions; check wording

In [5]:
norm = lambda s: re.sub(r'\s+', ' ', s.replace('’', "'").replace('‘', "'")).strip().lower()

colmap = {}  # export header -> variable code
for h in raw.columns:
    if norm(h) == 'timestamp':
        colmap[h] = 'timestamp'
    elif norm(h) == 'consent':
        colmap[h] = 'consent'
    else:
        colmap[h] = qcode(h) or re.sub(r'\W+', '_', h).strip('_').lower()
dups = pd.Series(colmap).value_counts()
assert (dups == 1).all(), f'Two export columns map to the same code: {dups[dups > 1].to_dict()}'

wording = []
for h, c in colmap.items():
    f = FORM.get(c)
    wording.append({'code': c, 'export_header': h, 'form_wording': f['title'] if f else '',
                    'in_form_script': bool(f), 'wording_matches': bool(f) and norm(f['title']) == norm(h)})
wording = pd.DataFrame(wording)
for _, r in wording[~wording.in_form_script & ~wording.code.isin(['timestamp'])].iterrows():
    log(1, 'column_not_in_form_script', r.code, detail=f'Export column "{r.export_header}" has no item in create_uber_form.gs; kept as-is')
for _, r in wording[wording.in_form_script & ~wording.wording_matches].iterrows():
    log(1, 'wording_differs', r.code, detail=f'Export header "{r.export_header}" != .gs "{r.form_wording}". '
        'Export header = live form title at export; which wording each respondent saw cannot be verified.')
missing_in_export = [c for c in FORM if c not in colmap.values()]
for c in missing_in_export:
    log(1, 'form_item_missing_from_export', c, detail=FORM[c]['title'])

print('Columns not in form script:', wording.loc[~wording.in_form_script, 'export_header'].tolist())
print('Form items missing from export:', missing_in_export)
display(wording[wording.in_form_script & ~wording.wording_matches][['code', 'export_header', 'form_wording']])

Columns not in form script: ['Timestamp', 'Consent']
Form items missing from export: []


,code,export_header,form_wording
9,Q5a,Q5a. Is the app showing exact location of yours or near-by places?...,"Q5a. What worked well, or what information was missing or confusing?"
26,Q15,Q15. Would you like to provide any additional feedback about Uber ...,Q15. Would you like to provide any additional feedback about Uber?


### 1.5 Respondent groups, eligibility and missingness

In [6]:
S = raw.rename(columns=colmap).apply(lambda s: s.str.strip())
ws_changed = int((raw.rename(columns=colmap) != S).sum().sum())
if ws_changed:
    log(2, 'strip_cell_whitespace', rows=ws_changed, detail='Leading/trailing spaces removed from cell values (text otherwise unchanged)')

resp_id = pd.Series([f'R{i + 1:02d}' for i in range(len(S))], index=S.index)
group = np.select([S['Q1'].isin(UBER_USER_CHOICES), S['Q1'].isin(NON_USER_CHOICES)], ['uber_user', 'non_uber'], 'unknown')
group = pd.Series(group, index=S.index)
if (group == 'unknown').any():
    log(1, 'q1_unrecognised', 'Q1', (group == 'unknown').sum(), 'Q1 blank or not a form option -> group "unknown"',
        ','.join(resp_id[group == 'unknown']))
print(group.value_counts().to_string())

def eligible_mask(c):
    e = FORM[c]['eligible'] if c in FORM else 'all'
    return pd.Series(True, index=S.index) if e == 'all' else (group == e)

miss = []
for c in S.columns:
    el, ans = eligible_mask(c), S[c] != ''
    miss.append({'code': c, 'type': FORM[c]['qtype'] if c in FORM else 'meta', 'required': FORM.get(c, {}).get('required', False),
                 'eligible_base': FORM[c]['eligible'] if c in FORM else 'all', 'n_eligible': int(el.sum()),
                 'n_answered': int((el & ans).sum()), 'n_missing_among_eligible': int((el & ~ans).sum()),
                 'n_answered_by_ineligible': int((~el & ans).sum())})
miss = pd.DataFrame(miss)
miss['pct_missing'] = (100 * miss.n_missing_among_eligible / miss.n_eligible.replace(0, np.nan)).round(1)
display(miss)

for _, r in miss.iterrows():
    if r.required and r.n_missing_among_eligible:
        ids = resp_id[eligible_mask(r.code) & (S[r.code] == '')]
        log(1, 'required_answer_missing', r.code, r.n_missing_among_eligible, 'Required item blank for eligible respondent; kept as missing', ','.join(ids))
    if r.n_answered_by_ineligible:
        ids = resp_id[~eligible_mask(r.code) & (S[r.code] != '')]
        log(1, 'answered_by_ineligible', r.code, r.n_answered_by_ineligible, 'Answer present for respondent outside branch; kept, excluded from denominators', ','.join(ids))
if 'consent' in S and (S['consent'] == '').all():
    log(1, 'consent_column_empty', 'consent', len(S), 'Consent checkbox was replaced by an implied-consent line in the form '
        'description (commit d24ac58); column is a leftover and is empty for all rows. Kept, not used.')

uber_user    26
non_uber      1


,code,type,required,eligible_base,n_eligible,n_answered,n_missing_among_eligible,n_answered_by_ineligible,pct_missing
0,timestamp,meta,False,all,27,27,0,0,0.0
1,consent,meta,False,all,27,0,27,0,100.0
2,Q1,single,True,all,27,27,0,0,0.0
3,Q2,scale,True,uber_user,26,26,0,0,0.0
4,Q2a,text,False,uber_user,26,21,5,0,19.2
5,Q3,single,True,uber_user,26,26,0,0,0.0
6,Q4,multi,True,uber_user,26,26,0,0,0.0
7,Q4a,text,False,uber_user,26,0,26,0,100.0
8,Q5,single,True,uber_user,26,26,0,0,0.0
9,Q5a,text,False,uber_user,26,0,26,0,100.0


### 1.6 Duplicates and out-of-option values

In [7]:
content_cols = [c for c in S.columns if c not in ('timestamp',)]
dup_content = S.duplicated(subset=content_cols, keep=False) & (S[content_cols] != '').any(axis=1)
dup_ts = S['timestamp'].duplicated(keep=False) & (S['timestamp'] != '')
blank_rows = (S[content_cols] == '').all(axis=1)
print(f'Identical answers (excluding timestamp): {dup_content.sum()} rows | duplicate timestamps: {dup_ts.sum()} | blank rows: {blank_rows.sum()}')
if dup_content.any():
    log(1, 'duplicate_content', rows=dup_content.sum(), detail='Rows with identical answers flagged (not removed)', resp_ids=','.join(resp_id[dup_content]))
if blank_rows.any():
    log(1, 'blank_row', rows=blank_rows.sum(), detail='Row with no answers flagged (not removed)', resp_ids=','.join(resp_id[blank_rows]))

out_of_opt = []
for c, f in FORM.items():
    if c not in S or f['qtype'] not in ('single', 'scale'):
        continue
    bad = (S[c] != '') & ~S[c].isin(f['options'])
    for v, n in S.loc[bad, c].value_counts().items():
        out_of_opt.append({'code': c, 'value': v, 'n': n})
        log(1, 'value_not_in_form_options', c, n, f'"{v}" kept as-is', ','.join(resp_id[bad & (S[c] == v)]))
print('Single-select / scale values not in form options:', out_of_opt or 'none')

Identical answers (excluding timestamp): 0 rows | duplicate timestamps: 0 | blank rows: 0
Single-select / scale values not in form options: none


## Phase 2 — Cleaning & data dictionary
### 2.1 Multi-select parsing
Google exports checkbox answers joined with `", "`. Each known option is matched as a whole item (longest first),
so write-in "Other" text containing commas is not split. Unmatched remainder = `_other_text`.

In [8]:
def split_multi(val, options):
    rest, found = val, []
    for o in sorted(options, key=len, reverse=True):
        pat = r'(?:^|(?<=, ))' + re.escape(o) + r'(?=, |$)'
        if re.search(pat, rest):
            found.append(o)
            rest = re.sub(pat, '\x00', rest)
    other = ', '.join(p.strip(' ,') for p in rest.split('\x00') if p.strip(' ,'))
    return [o for o in options if o in found], other

slug = lambda s: re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', s.lower())).strip('_')[:40]

MULTI = [c for c, f in FORM.items() if f['qtype'] == 'multi' and c in S]
multi_vars = {}
derived = {}
for c in MULTI:
    opts, el, ans = FORM[c]['options'], eligible_mask(c), S[c] != ''
    parsed = S[c].apply(lambda v: split_multi(v, opts))
    base = el & ans  # denominator rows: eligible and answered
    multi_vars[c] = []
    for o in opts:
        v = f'{c}__{slug(o)}'
        derived[v] = pd.Series(np.where(base, parsed.apply(lambda p: int(o in p[0])), np.nan), index=S.index).astype('Int64')
        multi_vars[c].append((v, o))
    other = parsed.apply(lambda p: p[1])
    derived[f'{c}__other'] = pd.Series(np.where(base, (other != '').astype(int), np.nan), index=S.index).astype('Int64')
    derived[f'{c}_other_text'] = other
    derived[f'{c}_n_selected'] = pd.Series(np.where(ans, parsed.apply(lambda p: len(p[0]) + int(p[1] != '')), np.nan), index=S.index).astype('Int64')
    n_other = int((other != '').sum())
    if n_other:
        tag = 'write_in_other_text' if FORM[c]['other_allowed'] else 'unmatched_text_in_multiselect'
        log(2, tag, c, n_other, 'Text not matching a form option moved to _other_text (original column unchanged)',
            ','.join(resp_id[other != '']))
print({c: len(v) for c, v in multi_vars.items()})

{'Q4': 9, 'Q6': 9, 'Q7': 8, 'Q9': 8}


### 2.2 Scale, timestamps, non-answers in open text

In [9]:
derived['Q2_rating'] = pd.to_numeric(S['Q2'].where(S['Q2'] != ''), errors='coerce')
bad_rating = (S['Q2'] != '') & ~derived['Q2_rating'].between(1, 5)
if bad_rating.any():
    log(2, 'rating_out_of_range', 'Q2', bad_rating.sum(), 'Set to NaN in Q2_rating; original kept in Q2', ','.join(resp_id[bad_rating]))

ts = pd.to_datetime(S['timestamp'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
if ts.isna().any():
    ts = ts.fillna(pd.to_datetime(S['timestamp'], errors='coerce'))
if ts.isna().any():
    log(2, 'timestamp_unparsed', 'timestamp', ts.isna().sum(), 'Kept original string', ','.join(resp_id[ts.isna()]))
derived['timestamp_parsed'] = ts

# Explicit "no content" tokens only. Words like "No"/"Nothing" are kept as answers (meaning depends on the question).
NONANSWER = {'na', 'n/a', 'n.a.', 'nil', 'none', '-', '--', '.', 'not applicable'}
TEXT = [c for c, f in FORM.items() if f['qtype'] == 'text' and c in S]
for c in TEXT:
    flag = S[c].str.lower().str.strip(' .').isin({t.strip(' .') for t in NONANSWER}) & (S[c] != '')
    derived[f'{c}_nonanswer'] = flag.astype(int).where(S[c] != '')
    if flag.any():
        log(2, 'open_text_nonanswer_flagged', c, flag.sum(), 'NA-style text flagged in _nonanswer; original text kept', ','.join(resp_id[flag]))

### 2.3 Consistency flags (flagged, never altered)
- An exclusive option ("no difficulty" / "not applicable") ticked together with substantive options.
- Q3 = "No" (never booked from a crowded place) but difficulties reported in Q4/Q6.

In [10]:
D = pd.DataFrame(derived, index=S.index)
flags = {}
EXCLUSIVE = [('Q4', 'I have not experienced such difficulty'), ('Q6', 'No difficulties'),
             ('Q7', 'Not applicable'), ('Q9', 'Not applicable'), ('Q9', 'No significant inconvenience')]
for c, o in EXCLUSIVE:
    if c not in MULTI or o not in FORM[c]['options']:
        continue
    v = f'{c}__{slug(o)}'
    f = (D[v] == 1) & (D[f'{c}_n_selected'] > 1)
    name = f'flag_{c}_{slug(o)[:20]}_plus_other'
    flags[name] = f.fillna(False).astype(bool)
    if flags[name].any():
        log(2, 'contradictory_multiselect', c, flags[name].sum(), f'"{o}" selected together with other options; flagged, not changed',
            ','.join(resp_id[flags[name]]))

no_crowded = S['Q3'] == 'No'
q4_loc = sum(D[v].fillna(0) for v, o in multi_vars['Q4'] if o != 'I have not experienced such difficulty') + D['Q4__other'].fillna(0) > 0
q6_diff = sum(D[v].fillna(0) for v, o in multi_vars['Q6'] if o != 'No difficulties') + D['Q6__other'].fillna(0) > 0
flags['flag_Q3_no_but_difficulty_reported'] = no_crowded & (q4_loc | q6_diff)
if flags['flag_Q3_no_but_difficulty_reported'].any():
    log(2, 'q3_no_but_difficulty', 'Q3', flags['flag_Q3_no_but_difficulty_reported'].sum(),
        'Q3="No" yet Q4/Q6 report difficulty (Q6 wording is about crowded places generally); flagged only',
        ','.join(resp_id[flags['flag_Q3_no_but_difficulty_reported']]))
display(pd.Series({k: int(v.sum()) for k, v in flags.items()}, name='rows_flagged'))

flag_Q4_i_have_not_experienc_plus_other    1
flag_Q6_no_difficulties_plus_other         0
flag_Q7_not_applicable_plus_other          0
flag_Q9_not_applicable_plus_other          0
flag_Q9_no_significant_incon_plus_other    2
flag_Q3_no_but_difficulty_reported         2
Name: rows_flagged, dtype: int64

### 2.4 Suspected test submissions — **pending researcher confirmation**
Identified by reading the response sheet: free-text fields contain keyboard strings (e.g. "fghj", "vdsa", "wefg").
They stay in the cleaned file with `flag_suspected_test = True` and `include_in_analysis = False`.
To include them, set `EXCLUDE_SUSPECTED_TESTS = False`. Their full text is printed below for review.

In [11]:
SUSPECTED_TEST_TIMESTAMPS = ['2026-10-14 10:04:58', '2026-10-14 10:07:29', '2026-10-14 10:12:29']
EXCLUDE_SUSPECTED_TESTS = True

ts_sec = ts.dt.round('s')  # export carries milliseconds; listed times are whole seconds
susp = ts_sec.isin(pd.to_datetime(SUSPECTED_TEST_TIMESTAMPS))
not_found = set(pd.to_datetime(SUSPECTED_TEST_TIMESTAMPS)) - set(ts_sec[susp])
if not_found:
    log(2, 'suspected_test_not_found', 'timestamp', len(not_found), f'Listed timestamps absent from export: {sorted(map(str, not_found))}')
if susp.any():
    log(2, 'suspected_test_flagged', rows=susp.sum(), resp_ids=','.join(resp_id[susp]),
        detail='Keyboard-string free text; flagged, kept in file; ' + ('excluded from analysis base' if EXCLUDE_SUSPECTED_TESTS else 'included')
               + ' (PENDING researcher confirmation)')
text_view = S.loc[susp, ['timestamp', 'Q1'] + [c for c in TEXT if c in S]].copy()
text_view.insert(0, 'resp_id', resp_id[susp])
display(text_view)

flags['flag_suspected_test'] = susp
flags['flag_duplicate_content'] = dup_content
flags['flag_blank_row'] = blank_rows
include = ~blank_rows & ~(susp & EXCLUDE_SUSPECTED_TESTS)
print(f'Rows in file: {len(S)} | include_in_analysis: {include.sum()} | uber users included: {(include & (group == "uber_user")).sum()}')

,resp_id,timestamp,Q1,Q2a,Q4a,Q5a,Q6a,Q7a,Q8a,Q9a,Q10a,Q11a,Q12,Q13,Q14a,Q15,Q14a_nonuber
0,R01,2026-10-14 10:04:57.567000,"No, I have never used Uber",,,,,,,,,,,,,,fghj
1,R02,2026-10-14 10:07:29.236000,"Yes, occasionally",,,,vdsa,,sav,,ewg,egwg,ewh,ewh,rhh,HW,
2,R03,2026-10-14 10:12:28.659000,"Yes, frequently",sdd,,,wefg,,dcevw,,ew,ef,ewf,efg,ef,wef,


Rows in file: 27 | include_in_analysis: 24 | uber users included: 24


### 2.5 Assemble cleaned dataset, verify counts, data dictionary

In [12]:
F = pd.DataFrame(flags, index=S.index)
meta = pd.DataFrame({'resp_id': resp_id, 'sheet_row': np.arange(2, len(S) + 2), 'respondent_group': group,
                     'include_in_analysis': include}, index=S.index)
def source_of(v):
    if v == 'timestamp_parsed':
        return 'timestamp'
    return re.match(r'(Q\d+[a-z]?(?:_nonuber)?)', v).group(1)
ordered = []
for c in S.columns:
    ordered.append(S[[c]])
    extra = [v for v in D.columns if source_of(v) == c]
    if extra:
        ordered.append(D[extra])
clean = pd.concat([meta, F] + ordered, axis=1)
assert len(clean) == len(raw), 'Row count changed — no record may be dropped'
assert clean.columns.is_unique

# Verify: parsed option counts vs naive substring count (what the Sheet's Summary tab COUNTIF "*option*" shows)
verify = []
for c in MULTI:
    for v, o in multi_vars[c]:
        parsed_n = int(clean[v].sum())
        naive_n = int(S[c].str.contains(re.escape(o), regex=True).sum())
        verify.append({'code': c, 'option': o, 'parsed_count_all_rows': parsed_n, 'substring_count_all_rows': naive_n, 'match': parsed_n == naive_n})
verify = pd.DataFrame(verify)
for _, r in verify[~verify.match].iterrows():
    log(2, 'count_check_mismatch', r.code, abs(r.parsed_count_all_rows - r.substring_count_all_rows),
        f'"{r.option}": parsed {r.parsed_count_all_rows} vs substring {r.substring_count_all_rows} (substring may also hit write-in text)')
print('Count check mismatches:', int((~verify.match).sum()), 'of', len(verify))

clean.to_csv(CLEAN / 'Cleaned_Primary_Research.csv', index=False, encoding='utf-8-sig')
print('Cleaned shape:', clean.shape)

Count check mismatches: 0 of 34
Cleaned shape: (27, 104)


In [13]:
dd = []
for c in S.columns:
    f = FORM.get(c, {})
    m = miss.set_index('code').loc[c]
    hdr = wording.set_index('code').loc[c, 'export_header']
    dd.append({'variable': c, 'derived_from': '', 'export_header': hdr, 'form_wording': f.get('title', ''),
               'type': f.get('qtype', 'meta'), 'options': ' | '.join(f.get('options', [])), 'required': f.get('required', ''),
               'other_allowed': f.get('other_allowed', ''), 'eligible_base': m.eligible_base, 'n_eligible': m.n_eligible,
               'n_answered': m.n_answered, 'notes': 'original text (whitespace-trimmed)'})
notes = {'__other': '1 = write-in/unmatched text present; NA = not eligible or not answered',
         '_other_text': 'write-in text not matching a form option', '_n_selected': 'options ticked incl. write-in',
         '_nonanswer': '1 = NA-style text (na, n/a, none, -, not applicable)', '_rating': 'numeric 1-5', '_parsed': 'datetime'}
for v in D.columns:
    src = source_of(v)
    opt = next((o for c in MULTI for vv, o in multi_vars[c] if vv == v), None)
    note = f'1 = selected "{opt}"; 0 = answered, not selected; NA = not eligible or not answered' if opt else \
        next((n for k, n in notes.items() if v.endswith(k)), '')
    dd.append({'variable': v, 'derived_from': src, 'type': 'derived', 'notes': note,
               'n_answered': int(D[v].notna().sum() if D[v].dtype != object else (D[v] != '').sum())})
for v in list(F.columns) + ['resp_id', 'sheet_row', 'respondent_group', 'include_in_analysis']:
    dd.append({'variable': v, 'type': 'flag/meta', 'n_answered': int(clean[v].sum()) if clean[v].dtype == bool else len(clean),
               'notes': 'sheet_row = row number in the Google Sheet (header = row 1)' if v == 'sheet_row' else
                        'respondent_group from Q1 branching' if v == 'respondent_group' else ''})
dd = pd.DataFrame(dd)
dd.to_csv(CLEAN / 'Data_Dictionary.csv', index=False, encoding='utf-8-sig')
display(dd[dd.type != 'derived'][['variable', 'type', 'eligible_base', 'n_eligible', 'n_answered', 'form_wording']])

,variable,type,eligible_base,n_eligible,n_answered,form_wording
0,timestamp,meta,all,27.0,27,
1,consent,meta,all,27.0,0,
2,Q1,single,all,27.0,27,Q1. Have you used Uber for booking a ride?
3,Q2,scale,uber_user,26.0,26,Q2. How would you rate your overall experience with Uber?
4,Q2a,text,uber_user,26.0,21,Q2a. What is the main reason for your rating?
5,Q3,single,uber_user,26.0,26,Q3. Have you booked an Uber from a crowded or complex location?
6,Q4,multi,uber_user,26.0,26,Q4. Where have you experienced difficulty finding or meeting your ...
7,Q4a,text,uber_user,26.0,0,"Q4a. If you remember, mention the location type or entrance where ..."
8,Q5,single,uber_user,26.0,26,Q5. How helpful is Uber's app in showing your pickup location and ...
9,Q5a,text,uber_user,26.0,0,"Q5a. What worked well, or what information was missing or confusing?"


### 2.6 Reconcile provisional figures in `PROMPT.md`
Shown for both bases (all rows / `include_in_analysis`). Denominator = eligible respondents who answered that question.
Where a figure could come from two questions, both are shown.

In [14]:
BRIEF = [('Entrance/gate confusion', 13, 'Q6', 'Confusion about which entrance or gate to use'),
         ('Incorrect/inconvenient pickup pin', 12, 'Q6', 'Pickup pin showed an incorrect or inconvenient location'),
         ('Long waiting', 11, 'Q6', 'Long waiting time'),
         ('Driver unable to find exact location', 9, 'Q6', 'Driver could not find my exact location'),
         ('Driver did not communicate clearly', 8, 'Q7', 'Did not communicate clearly'),
         ('Driver asked rider to walk elsewhere', 8, 'Q6', 'Driver asked me to walk to another location'),
         ('Driver asked rider to walk elsewhere', 8, 'Q7', 'Asked me to move to another location'),
         ('Driver unable to access pickup point', 4, 'Q6', 'Driver could not access the pickup point'),
         ('Repeated calls/messages', 3, 'Q6', 'Repeated phone calls or messages'),
         ('Rider cancellation', 6, 'Q10', 'I cancelled the booking'),
         ('Driver cancellation', 4, 'Q10', 'Driver cancelled the booking'),
         ('Driver cancellation', 4, 'Q7', 'Cancelled the booking')]
rec = []
for label, prov, c, o in BRIEF:
    row = {'brief_item': label, 'provisional': prov, 'source_question': c, 'option': o, 'question_type': FORM[c]['qtype']}
    for base_name, base in [('all', pd.Series(True, index=S.index)), ('included', include)]:
        el = base & eligible_mask(c) & (S[c] != '')
        if FORM[c]['qtype'] == 'multi':
            n = int(clean.loc[el, f'{c}__{slug(o)}'].sum())
        else:
            n = int((S.loc[el, c] == o).sum())
        row[f'count_{base_name}'], row[f'denom_{base_name}'] = n, int(el.sum())
        row[f'pct_{base_name}'] = round(100 * n / el.sum(), 1) if el.sum() else np.nan
    row['matches_provisional_all'] = row['count_all'] == prov
    rec.append(row)
r_all, r_inc = derived['Q2_rating'], derived['Q2_rating'][include]
rec.append({'brief_item': 'Average Uber rating (1-5)', 'provisional': 3.7, 'source_question': 'Q2', 'option': 'mean', 'question_type': 'scale',
            'count_all': round(r_all.mean(), 2), 'denom_all': int(r_all.notna().sum()),
            'count_included': round(r_inc.mean(), 2), 'denom_included': int(r_inc.notna().sum()),
            'matches_provisional_all': round(r_all.mean(), 1) == 3.7})
rec = pd.DataFrame(rec)
rec.to_csv(CLEAN / 'Count_Reconciliation.csv', index=False, encoding='utf-8-sig')
display(rec)

,brief_item,provisional,source_question,option,question_type,count_all,denom_all,pct_all,count_included,denom_included,pct_included,matches_provisional_all
0,Entrance/gate confusion,13.0,Q6,Confusion about which entrance or gate to use,multi,13.00,26,50.0,13.00,24,54.2,True
1,Incorrect/inconvenient pickup pin,12.0,Q6,Pickup pin showed an incorrect or inconvenient location,multi,12.00,26,46.2,10.00,24,41.7,True
2,Long waiting,11.0,Q6,Long waiting time,multi,11.00,26,42.3,11.00,24,45.8,True
3,Driver unable to find exact location,9.0,Q6,Driver could not find my exact location,multi,9.00,26,34.6,9.00,24,37.5,True
4,Driver did not communicate clearly,8.0,Q7,Did not communicate clearly,multi,8.00,26,30.8,8.00,24,33.3,True
5,Driver asked rider to walk elsewhere,8.0,Q6,Driver asked me to walk to another location,multi,8.00,26,30.8,8.00,24,33.3,True
6,Driver asked rider to walk elsewhere,8.0,Q7,Asked me to move to another location,multi,7.00,26,26.9,6.00,24,25.0,False
7,Driver unable to access pickup point,4.0,Q6,Driver could not access the pickup point,multi,4.00,26,15.4,4.00,24,16.7,True
8,Repeated calls/messages,3.0,Q6,Repeated phone calls or messages,multi,3.00,26,11.5,3.00,24,12.5,True
9,Rider cancellation,6.0,Q10,I cancelled the booking,single,6.00,26,23.1,6.00,24,25.0,True


In [15]:
log_df = pd.DataFrame(LOG)
log_df.insert(0, 'log_id', range(1, len(log_df) + 1))
log_df.to_csv(CLEAN / 'Cleaning_Log.csv', index=False, encoding='utf-8-sig')
display(log_df)

for name in ['Cleaned_Primary_Research.csv', 'Data_Dictionary.csv', 'Cleaning_Log.csv', 'Count_Reconciliation.csv']:
    p = CLEAN / name
    print(f'{"OK " if p.exists() else "MISSING"} {p}  ({p.stat().st_size if p.exists() else 0} bytes)')

,log_id,phase,action,column,rows_affected,detail,resp_ids
0,1,1,load_input,,27,Uber Ride & Pickup Experience Survey (Responses).xlsx [sheet Form ...,
1,2,1,column_not_in_form_script,consent,0,"Export column ""Consent"" has no item in create_uber_form.gs; kept a...",
2,3,1,wording_differs,Q5a,0,"Export header ""Q5a. Is the app showing exact location of yours or ...",
3,4,1,wording_differs,Q15,0,"Export header ""Q15. Would you like to provide any additional feedb...",
4,5,2,strip_cell_whitespace,,42,Leading/trailing spaces removed from cell values (text otherwise u...,
5,6,1,consent_column_empty,consent,27,Consent checkbox was replaced by an implied-consent line in the fo...,
6,7,2,write_in_other_text,Q4,3,Text not matching a form option moved to _other_text (original col...,"R04,R16,R23"
7,8,2,open_text_nonanswer_flagged,Q11a,1,NA-style text flagged in _nonanswer; original text kept,R08
8,9,2,contradictory_multiselect,Q4,1,"""I have not experienced such difficulty"" selected together with ot...",R13
9,10,2,contradictory_multiselect,Q9,2,"""No significant inconvenience"" selected together with other option...","R17,R27"


OK  D:\Crop_python\IIMA\Airtribe\Uber\TMS\data\clean\Cleaned_Primary_Research.csv  (25901 bytes)
OK  D:\Crop_python\IIMA\Airtribe\Uber\TMS\data\clean\Data_Dictionary.csv  (16645 bytes)
OK  D:\Crop_python\IIMA\Airtribe\Uber\TMS\data\clean\Cleaning_Log.csv  (2047 bytes)
OK  D:\Crop_python\IIMA\Airtribe\Uber\TMS\data\clean\Count_Reconciliation.csv  (1513 bytes)


## Inference (descriptive, Phase 1-2 data)

**Base:** 24 Uber users in `include_in_analysis` (27 rows minus 1 non-user and 2 suspected test submissions, R02 and R03). Percentages use respondents who answered that question as the denominator. Multi-select shares overlap and must not be added together. This is a small convenience sample, so the findings are descriptive only: no causal or market-wide claims.

**1. Pickup difficulty is common in this sample.**
- 22 of 24 (91.7%) have booked from a crowded or complex location (Q3).
- Only 2 of 24 (8.3%) ticked "No difficulties" in Q6.
- Overall experience is moderate: mean 3.79, median 4, and no rating below 3 among included respondents (Q2).

**2. Location ambiguity is the most reported problem, ahead of waiting time (Q6, multi-select, n=24).**
- Entrance/gate confusion: 13 (54.2%).
- Long waiting time: 11 (45.8%).
- Incorrect or inconvenient pickup pin: 10 (41.7%).
- Driver could not find the exact location: 9 (37.5%).
- Driver asked rider to walk elsewhere: 8 (33.3%).
- Gate/pin confusion points to a "last-metre" problem, meaning the pin is roughly right but the exact meeting point is unclear.
- In Q5, 10 of 24 (41.7%) find the app "sometimes confusing or inaccurate" and only 3 (12.5%) find it "very helpful and accurate".

**3. Coordination falls back on manual driver-rider contact (Q7, multi-select, n=24).**
- Called to ask for directions: 11 (45.8%).
- Did not communicate clearly: 8 (33.3%).
- Appeared confused about the location: 7 (29.2%).
- Cancelled the booking: 7 (29.2%).
- Helped identify a suitable meeting point: 4 (16.7%).
- Constructive in-app help is the least common behaviour, so the app does little to resolve ambiguity.
- Driver behaviour in Q8 is mostly neutral or varies (13 of 24). Only 5 of 24 describe drivers as cooperative or very polite, and 6 as frustrated or impatient or as rude or unhelpful.

**4. The main cost to riders is time pressure, and a difficult pickup often ends in a cancellation or delay.**
- 15 of 24 (62.5%) were "worried about being late" (Q9). This is the largest emotional response, ahead of frustrated (6) and anxious or stressed (6).
- Q10 is single-select, so these shares add up. Only 3 of 24 (12.5%) met the driver without much delay. 9 (37.5%) met after a delay, 6 (25.0%) cancelled themselves, and 4 (16.7%) were cancelled on by the driver. So 10 of 24 (41.7%) ended in a cancellation.
- Mean Q2 rating is higher for riders who met the driver without much delay (4.67, n=3) than for those who cancelled (3.33, n=6). The groups are too small for a reliable comparison, so treat the gap as directional only.

**5. Reported extra-money requests are a separate issue from pickup coordination.**
- 14 of 24 (58.3%) said a driver had asked for extra money (Q11).
- Stated reasons (Q11a) are mostly toll, fuel, traffic, distance and peak hours. They are not about pickup location, so keep this out of the pickup-friction claim.
- Several Q10a answers (cancellations) also mention extra money or drivers declining the location.

**6. Competitor signal.**
- 14 of 24 (58.3%) prefer Rapido (8) or Ola (6) over Uber, against 4 (16.7%) who prefer Uber and 3 (12.5%) with no preference. A further 3 chose "Other" (Q14).
- This is stated preference only. The survey does not show why, or whether they switch.

**7. Improvement asks (Q13, 12 of 24 answered, qualitative).**
- Accurate pickup pins and exact location (R06, R16, R21).
- Driver training (R07, R12).
- More driver supply (R07, R10, R24).
- Alerting drivers to crowded locations (R24).
- A fuller thematic coding of Q6a, Q8a, Q12 and Q13 is still pending in a later phase.

**Changes to the provisional figures in `PROMPT.md`:** excluding the two test submissions changes some counts. Incorrect or inconvenient pickup pin falls from 12 to 10 and "asked me to move" (Q7) from 7 to 6. Mean rating moves from 3.73 to 3.79. Entrance/gate confusion (13), long waiting (11) and the other Q6 counts are unchanged but now sit over a denominator of 24 instead of 26. See `data/clean/Count_Reconciliation.csv`.

**Caveats**
- n=24, self-selected, with an unknown mix of cities.
- 3 respondents were flagged for contradictory multi-select answers (R13, R17, R27).
- 2 respondents answered Q3 "No" but still reported difficulty (R06, R19).
- Open-text fields Q4a, Q5a, Q7a and Q9a have no responses in the export.

## Phase 3 - Charts and heatmaps
Base: included Uber users. Multi-select shares overlap. Heatmaps are respondent-level cross-tabs of a small sample: descriptive only.

In [ ]:
# Phase 3 charts (descriptive; base = included Uber users)
import textwrap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
CHARTS = ROOT / 'charts'; CHARTS.mkdir(exist_ok=True)
INK, MUTED, BAR, GRID = '#1f2933', '#6b7785', '#2a6f97', '#e3e7eb'
plt.rcParams.update({'font.size': 10, 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED, 'xtick.color': MUTED,
                     'ytick.color': INK, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110})
base = clean[clean['include_in_analysis'] & (clean['respondent_group'] == 'uber_user')]
N = len(base)
wrap = lambda s, w=34: '
'.join(textwrap.wrap(s, w))

def hbar(labels, counts, title, fname, note=''):
    order = np.argsort(counts)
    labels, counts = [labels[i] for i in order], [counts[i] for i in order]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(labels) + 1.6))
    ax.barh([wrap(l) for l in labels], counts, color=BAR, height=0.6)
    for i, n in enumerate(counts):
        ax.text(n + 0.2, i, f'{n} ({100 * n / N:.0f}%)', va='center', color=INK, fontsize=9)
    ax.set_xlim(0, max(counts) * 1.25); ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
    ax.set_xlabel(f'Respondents (n={N}){note}')
    ax.set_title(title, loc='left', color=INK, fontsize=11, fontweight='bold')
    fig.tight_layout(); fig.savefig(CHARTS / fname, dpi=160); plt.close(fig)

def multi_counts(q):
    opts = MULTI[q]
    return opts, [int(base[f'{q}__{slug(o)}'].sum()) for o in opts]

for q, title, fname in [('Q6', 'Q6. Difficulties meeting the driver (multi-select)', 'q6_difficulties.png'),
                        ('Q4', 'Q4. Where pickup difficulty occurred (multi-select)', 'q4_locations.png'),
                        ('Q7', 'Q7. How the driver coordinated (multi-select)', 'q7_driver_coordination.png'),
                        ('Q9', 'Q9. How riders felt (multi-select)', 'q9_feelings.png')]:
    o, c = multi_counts(q)
    hbar(o, c, title, fname, note='; shares overlap, do not sum')
for q, title, fname in [('Q10', 'Q10. Outcome of the difficult pickup (single-select)', 'q10_outcomes.png'),
                        ('Q5', 'Q5. How helpful is the app at showing pickup (single-select)', 'q5_app_helpfulness.png'),
                        ('Q2', 'Q2. Overall rating (1-5)', 'q2_rating.png')]:
    vc = base[q if q != 'Q2' else 'Q2_rating'].value_counts()
    hbar([str(int(i)) if q == 'Q2' else i for i in vc.index], [int(v) for v in vc.values], title, fname)

def heat(rows, cols, rq, cq, title, fname):
    ro, co = MULTI[rq], (MULTI[cq] if cq in MULTI else sorted(base[cq].dropna().unique()))
    M = np.array([[int((base[f'{rq}__{slug(r)}'].astype(bool) & (base[f'{cq}__{slug(c)}'].astype(bool) if cq in MULTI else base[cq] == c)).sum())
                   for c in co] for r in ro])
    rn = [int(base[f'{rq}__{slug(r)}'].sum()) for r in ro]
    ri = [i for i in np.argsort(rn)[::-1] if rn[i] > 0 and not re.match(r'(?i)i have not|no difficult|not applicable', ro[i])]
    ci = [j for j, c in enumerate(co) if M[:, j].sum() > 0 and not re.match(r'(?i)no difficult|not applicable', str(c))]
    M2 = M[np.ix_(ri, ci)]
    fig, ax = plt.subplots(figsize=(1.0 * len(ci) + 3.5, 0.55 * len(ri) + 2.4))
    im = ax.imshow(M2, cmap='Blues', vmin=0, aspect='auto')
    ax.set_xticks(range(len(ci))); ax.set_xticklabels([wrap(str(co[j]), 16) for j in ci], fontsize=8)
    ax.set_yticks(range(len(ri))); ax.set_yticklabels([f'{wrap(ro[i], 28)}  (n={rn[i]})' for i in ri], fontsize=8)
    ax.xaxis.tick_top()
    for a in range(len(ri)):
        for b in range(len(ci)):
            v = M2[a, b]; ax.text(b, a, v, ha='center', va='center', fontsize=9,
                                  color='white' if v > M2.max() * 0.6 else INK)
    ax.set_title(title, loc='left', color=INK, fontsize=11, fontweight='bold', pad=48)
    ax.set_xlabel(f'Cells = respondents ticking both (n={N}). Small cells (<3) are anecdotal; no causal reading.', fontsize=8)
    fig.tight_layout(); fig.savefig(CHARTS / fname, dpi=160); plt.close(fig)

heat(None, None, 'Q4', 'Q6', 'Heatmap: location (Q4) x difficulty (Q6)', 'heatmap_location_x_difficulty.png')
heat(None, None, 'Q6', 'Q10', 'Heatmap: difficulty (Q6) x outcome (Q10)', 'heatmap_difficulty_x_outcome.png')
print(sorted(p.name for p in CHARTS.glob('*.png')))
